# fNIRS Signal Cleaning: ICA and Anti-Correlation Methods
### A deep-dive into artifact removal for single-sensor cerebral blood flow devices

**Context:** Temple's wearable sits on the temple and measures cerebral blood flow with one sensor.
Unlike lab fNIRS setups that use a short-separation channel to subtract scalp noise, a single wearable
sensor has no reference. This notebook implements the two main software approaches to solve that problem.

**What you'll learn:**
1. Why scalp hemodynamics contaminate fNIRS signals
2. How ICA separates mixed sources using wavelet-based virtual channels
3. How the Anti-Correlation (CBSI) method exploits HbO/HbR physiology
4. What each method removes and what it preserves
5. How to evaluate cleaning quality without ground truth

---


## 0. Setup and Imports

In [ ]:
import sys, warnings
sys.path.insert(0, '..')   # adjust if running from notebooks/ subfolder
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pywt
from scipy.stats import kurtosis
from sklearn.decomposition import FastICA

# Our pipeline modules
from data.generate import generate_dataset, generate_epoch, FNIRSConfig
from utils.preprocess import preprocess_dataset, PreprocConfig, bandpass_filter, baseline_correct
from utils.artifact_removal import (
    _wavelet_decompose,
    _identify_artifact_components,
    apply_ica_single_channel,
    apply_ica_multichannel,
    apply_cbsi,
    full_artifact_removal_pipeline,
    ICAConfig, CBSIConfig,
)

plt.rcParams.update({
    'figure.facecolor': '#0e1117',
    'axes.facecolor':   '#1a1d27',
    'axes.edgecolor':   '#444',
    'axes.labelcolor':  'white',
    'xtick.color':      'white',
    'ytick.color':      'white',
    'text.color':       'white',
    'grid.color':       '#333',
    'grid.linestyle':   '--',
    'grid.alpha':       0.4,
    'figure.dpi':       120,
})

HBO_COLOR = '#e74c3c'
HBR_COLOR = '#3498db'
CLEAN_HBO  = '#f39c12'
CLEAN_HBR  = '#2ecc71'

print("Setup complete.")

---
## Part 1: Understanding the Problem — Why Scalp Noise Contaminates fNIRS

fNIRS measures **changes in light absorption** through the skull. The light passes through:
1. **Scalp and skin** — where blood vessels respond to heart rate, blood pressure, breathing
2. **Skull** — mostly transparent to NIR light
3. **Brain cortex** — where neural activity drives *local* hemodynamic changes

Because the light path goes through scalp first, the sensor picks up:

| Source | Frequency | HbO/HbR behavior |
|---|---|---|
| Mayer waves (blood pressure) | ~0.10 Hz | Both increase together |
| Respiration | ~0.25 Hz | Both increase together |  
| Cardiac pulse | ~1.0 Hz | Both increase together |
| **Neural activation** | **0.01–0.1 Hz** | **HbO ↑, HbR ↓ (anti-correlated)** |

The key insight: **scalp noise makes HbO and HbR move together (co-correlated).
True neural signals make them move in opposite directions (anti-correlated).**


In [ ]:
# Generate a dataset and visualize raw vs clean signals
rng = np.random.default_rng(42)
cfg = FNIRSConfig(fs=10.0, n_channels=16, epoch_len=10.0, n_epochs_per_class=60)
raw = generate_dataset(cfg)
proc_cfg = PreprocConfig(fs=10.0)
proc = preprocess_dataset(raw, proc_cfg)

# Pick one MOTOR epoch for all demos
motor_idx = np.where(proc['y'] == 2)[0][0]
hbo_raw = proc['X_hbo'][motor_idx]   # (16, 100)
hbr_raw = proc['X_hbr'][motor_idx]

t = np.linspace(0, cfg.epoch_len, hbo_raw.shape[1])

# Show 3 channels
fig, axes = plt.subplots(3, 1, figsize=(13, 7), sharex=True)
fig.suptitle('Raw preprocessed fNIRS — MOTOR epoch (3 channels)', fontsize=13)

for i, ch in enumerate([0, 4, 8]):
    ax = axes[i]
    ax.plot(t, hbo_raw[ch], color=HBO_COLOR, linewidth=1.5, label='HbO (oxygenated Hb)')
    ax.plot(t, hbr_raw[ch], color=HBR_COLOR, linewidth=1.5, label='HbR (deoxygenated Hb)')
    corr = np.corrcoef(hbo_raw[ch], hbr_raw[ch])[0,1]
    ax.set_ylabel(f'Ch {ch}\nCorr={corr:.2f}', fontsize=9)
    ax.grid(True)
    if i == 0:
        ax.legend(loc='upper right', fontsize=9)

axes[-1].set_xlabel('Time (s)')
plt.tight_layout()
plt.show()

print(f"Mean HbO-HbR correlation across all channels: {np.mean([np.corrcoef(hbo_raw[c], hbr_raw[c])[0,1] for c in range(hbo_raw.shape[0])]):.3f}")
print("(Negative = anti-correlated = neural signal. Positive = co-correlated = systemic noise.)")

---
## Part 2: Independent Component Analysis (ICA)

### The cocktail party problem

Imagine being at a party with 4 people talking simultaneously, recorded by 4 microphones.
Each microphone picks up a *mixture* of all 4 voices. ICA's job: unmix them back into
4 independent source tracks.

For fNIRS: the "microphones" are our sensor channels, and the "voices" are:
- Neural hemodynamic response (what we want)
- Cardiac pulsation (~1 Hz)
- Respiratory artifact (~0.25 Hz)  
- Mayer wave artifact (~0.10 Hz)
- Motion artifacts (broadband)

### The single-channel challenge

Standard ICA requires **at least as many channels as sources** to separate them.
A single Temple sensor gives us 1 channel. Solution: **wavelet decomposition**
creates "virtual channels" by splitting the signal into frequency sub-bands.

```
Single fNIRS channel
        ↓
   DWT decomposition (db4 wavelet)
        ↓
  Frequency sub-bands (virtual channels):
    Band 0 (approx):  0.0  – 0.16 Hz  ← VLF, neural hemodynamics live here
    Band 1 (detail):  0.16 – 0.31 Hz  ← Mayer wave edge
    Band 2 (detail):  0.31 – 0.63 Hz  ← Respiratory
    Band 3 (detail):  0.63 – 1.25 Hz  ← Cardiac edge
        ↓
  FastICA on virtual channels
        ↓
  Kurtosis-based artifact identification
        ↓
  Remove artifact components, reconstruct
```


In [ ]:
# ── Step 1: Visualize wavelet decomposition ──
# Take channel 0 of our HbO signal and decompose it

sig_demo = hbo_raw[0]  # (100,)
wavelet = 'db4'
n_samples = len(sig_demo)
max_level = pywt.dwt_max_level(n_samples, wavelet)

coeffs = pywt.wavedec(sig_demo, wavelet, level=max_level)
print(f"Signal length: {n_samples} samples ({cfg.epoch_len}s at {cfg.fs}Hz)")
print(f"Max DWT level: {max_level}")
print(f"Number of sub-bands: {len(coeffs)} (1 approximation + {max_level} details)")

# Reconstruct each sub-band separately to visualize
sub_bands = []
band_names = []
freq_ranges = [
    f'0.0–{cfg.fs/2**(max_level+1):.2f} Hz (VLF/neural)',
    f'{cfg.fs/2**(max_level+1):.2f}–{cfg.fs/2**max_level:.2f} Hz (Mayer edge)',
    f'{cfg.fs/2**max_level:.2f}–{cfg.fs/2**(max_level-1):.2f} Hz (Respiratory)',
    f'{cfg.fs/2**(max_level-1):.2f}–{cfg.fs/2**(max_level-2):.2f} Hz (Cardiac edge)',
]

for i, c in enumerate(coeffs):
    zero_c = [np.zeros_like(cc) for cc in coeffs]
    zero_c[i] = c
    rec = pywt.waverec(zero_c, wavelet)[:n_samples]
    sub_bands.append(rec)
    label = freq_ranges[i] if i < len(freq_ranges) else f'Detail {i}'
    band_names.append(f'Band {i}: {label}')

fig, axes = plt.subplots(len(sub_bands) + 1, 1, figsize=(13, 10), sharex=True)
fig.suptitle('DWT Sub-band Decomposition of fNIRS Channel 0\n(Each row = one virtual channel for ICA)', fontsize=12)

axes[0].plot(t, sig_demo, color=HBO_COLOR, linewidth=1.5)
axes[0].set_ylabel('Original\nHbO', fontsize=8)
axes[0].grid(True)

for i, (band, name) in enumerate(zip(sub_bands, band_names)):
    ax = axes[i+1]
    ax.plot(t, band, color='#9b59b6', linewidth=1.2)
    ax.set_ylabel(f'Sub-band {i}', fontsize=8)
    ax.set_title(name, fontsize=8, pad=2)
    ax.grid(True)

axes[-1].set_xlabel('Time (s)')
plt.tight_layout()
plt.show()

In [ ]:
# ── Step 2: Run FastICA on virtual channels ──

ica_cfg = ICAConfig(fs=cfg.fs)
pseudo = _wavelet_decompose(sig_demo, ica_cfg.n_components, ica_cfg.fs)
X = pseudo.T  # (n_samples, n_components)

ica = FastICA(
    n_components=ica_cfg.n_ica_components,
    max_iter=500, random_state=42, whiten='unit-variance'
)
comps_T = ica.fit_transform(X)
comps = comps_T.T  # (n_components, n_samples)

kurt_vals = kurtosis(comps, axis=1)
artifact_mask, _ = _identify_artifact_components(comps, ica_cfg.kurtosis_threshold)

print("ICA components extracted:")
print(f"{'Component':>10} {'Kurtosis':>12} {'Classification':>18}")
print("-" * 44)
for i, (k, is_art) in enumerate(zip(kurt_vals, artifact_mask)):
    label = '🔴 ARTIFACT (remove)' if is_art else '🟢 signal  (keep)'
    print(f"   IC {i:>2}     {k:>10.2f}    {label}")

print(f"\nThreshold: |kurtosis| > {ica_cfg.kurtosis_threshold}")
print("Low kurtosis → Gaussian/smooth → neural hemodynamics")
print("High kurtosis → spiky/impulsive → motion or cardiac artifact")

In [ ]:
# ── Step 3: Visualize ICA components and their kurtosis ──

fig = plt.figure(figsize=(14, 8))
gs = gridspec.GridSpec(2, 2, figure=fig)
fig.suptitle('FastICA Components and Artifact Identification', fontsize=13)

# Left: time-domain components
ax_time = fig.add_subplot(gs[:, 0])
for i, comp in enumerate(comps):
    color = '#e74c3c' if artifact_mask[i] else '#2ecc71'
    label = f'IC{i} (kurt={kurt_vals[i]:.1f}) {"← ARTIFACT" if artifact_mask[i] else "← keep"}'
    ax_time.plot(t, comp + i * 3, color=color, linewidth=1.3, label=label)
ax_time.set_xlabel('Time (s)')
ax_time.set_ylabel('Component amplitude (offset for clarity)')
ax_time.set_title('ICA Components (time domain)', fontsize=11)
ax_time.legend(fontsize=8, loc='upper right')
ax_time.grid(True)

# Top right: kurtosis bar chart
ax_kurt = fig.add_subplot(gs[0, 1])
colors = ['#e74c3c' if m else '#2ecc71' for m in artifact_mask]
bars = ax_kurt.bar([f'IC{i}' for i in range(len(kurt_vals))], np.abs(kurt_vals), color=colors)
ax_kurt.axhline(ica_cfg.kurtosis_threshold, color='white', linestyle='--', linewidth=1.5, label=f'Threshold={ica_cfg.kurtosis_threshold}')
ax_kurt.set_ylabel('|Kurtosis|')
ax_kurt.set_title('Artifact Detection by Kurtosis', fontsize=11)
ax_kurt.legend(fontsize=9)
ax_kurt.grid(True, axis='y')

# Bottom right: before/after comparison
ax_ba = fig.add_subplot(gs[1, 1])
clean_sig, diag = apply_ica_single_channel(sig_demo, ica_cfg, return_diagnostics=True)
ax_ba.plot(t, sig_demo, color=HBO_COLOR, linewidth=1.5, label=f'Raw HbO (std={sig_demo.std():.4f})', alpha=0.7)
ax_ba.plot(t, clean_sig, color=CLEAN_HBO, linewidth=1.5, label=f'After ICA (std={clean_sig.std():.4f})', linestyle='--')
ax_ba.set_xlabel('Time (s)')
ax_ba.set_ylabel('HbO amplitude')
ax_ba.set_title(f'Before vs After ICA ({diag["n_removed"]} components removed)', fontsize=11)
ax_ba.legend(fontsize=9)
ax_ba.grid(True)

plt.tight_layout()
plt.show()

---
## Part 3: The Anti-Correlation (CBSI) Method

### The physiological constraint

Every neuroscience textbook describes the **neurovascular coupling** response:
- Neurons fire → local metabolic demand increases → blood flow redirected to the area
- **Oxygenated hemoglobin (HbO) floods in** → HbO signal rises
- **Deoxygenated hemoglobin (HbR) is washed out** → HbR signal falls

This **anti-correlation** is so reliable that Cui et al. (2010) turned it into
a cleaning algorithm called **CBSI: Correlation-Based Signal Improvement**.

### The math

Let α = std(HbO) / std(HbR)  ← scaling factor to account for amplitude difference

```
Corrected HbO = 0.5 × (HbO − α × HbR)
Corrected HbR = 0.5 × (HbR − (1/α) × HbO)
```

**Why this works:**
- For **systemic noise**: HbO ≈ α × HbR (they scale together) → subtraction cancels noise
- For **neural signal**: HbO ≈ −α × HbR (they're anti-correlated) → subtraction doubles it

After CBSI, the corrected HbO and HbR are **perfectly anti-correlated by construction**
(correlation = −1.0). This is a mathematical guarantee, not an empirical result.


In [ ]:
# ── Demonstrate CBSI step by step on one channel ──

ch = 0
h = hbo_raw[ch]
r = hbr_raw[ch]

# Step 1: compute alpha
std_h = h.std()
std_r = r.std()
alpha = float(np.clip(std_h / (std_r + 1e-10), 0.1, 10.0))

# Step 2: apply correction
h_corr = 0.5 * (h - alpha * r)
r_corr = 0.5 * (r - (1.0 / alpha) * h)

# Step 3: compute correlations
corr_before = np.corrcoef(h, r)[0, 1]
corr_after  = np.corrcoef(h_corr, r_corr)[0, 1]

print("CBSI — Channel 0")
print(f"  α (scaling factor) = std(HbO) / std(HbR) = {std_h:.4f} / {std_r:.4f} = {alpha:.3f}")
print(f"  HbO-HbR correlation BEFORE: {corr_before:+.3f}")
print(f"  HbO-HbR correlation AFTER:  {corr_after:+.3f}  (forced to -1.0 by construction)")

fig, axes = plt.subplots(2, 2, figsize=(14, 7))
fig.suptitle('CBSI Anti-Correlation Method — Channel 0', fontsize=13)

# Before: time domain
ax = axes[0, 0]
ax.plot(t, h, color=HBO_COLOR, linewidth=1.5, label=f'HbO')
ax.plot(t, r, color=HBR_COLOR, linewidth=1.5, label=f'HbR')
ax.set_title(f'Before CBSI (corr={corr_before:+.3f})', fontsize=10)
ax.set_xlabel('Time (s)'); ax.legend(); ax.grid(True)

# After: time domain  
ax = axes[0, 1]
ax.plot(t, h_corr, color=CLEAN_HBO, linewidth=1.5, label='HbO corrected')
ax.plot(t, r_corr, color=CLEAN_HBR, linewidth=1.5, label='HbR corrected')
ax.set_title(f'After CBSI (corr={corr_after:+.3f})', fontsize=10)
ax.set_xlabel('Time (s)'); ax.legend(); ax.grid(True)

# Before: scatter plot
ax = axes[1, 0]
ax.scatter(h, r, color='#9b59b6', alpha=0.6, s=15)
ax.set_xlabel('HbO'); ax.set_ylabel('HbR')
ax.set_title(f'HbO vs HbR before (corr={corr_before:+.3f})', fontsize=10)
# Add trend line
z = np.polyfit(h, r, 1)
p = np.poly1d(z)
x_line = np.linspace(h.min(), h.max(), 50)
ax.plot(x_line, p(x_line), 'w--', linewidth=1.5)
ax.grid(True)

# After: scatter plot
ax = axes[1, 1]
ax.scatter(h_corr, r_corr, color='#2ecc71', alpha=0.6, s=15)
ax.set_xlabel('HbO corrected'); ax.set_ylabel('HbR corrected')
ax.set_title(f'HbO vs HbR after (corr={corr_after:+.3f})', fontsize=10)
z2 = np.polyfit(h_corr, r_corr, 1)
p2 = np.poly1d(z2)
x2 = np.linspace(h_corr.min(), h_corr.max(), 50)
ax.plot(x2, p2(x2), 'w--', linewidth=1.5)
ax.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# ── Show CBSI effect across all classes: does it preserve the class difference? ──

fig, axes = plt.subplots(3, 2, figsize=(14, 9))
fig.suptitle('CBSI Applied to All Three Mental States\n(does cleaning preserve the class signal?)', fontsize=12)

class_names = {0: 'REST', 1: 'MENTAL', 2: 'MOTOR'}
colors_class = {0: '#95a5a6', 1: '#e67e22', 2: '#2ecc71'}

for row, label in enumerate([0, 1, 2]):
    idx = np.where(proc['y'] == label)[0][0]
    h = proc['X_hbo'][idx][0]
    r = proc['X_hbr'][idx][0]
    h_c, r_c, diag = apply_cbsi(
        proc['X_hbo'][idx], proc['X_hbr'][idx], return_diagnostics=True
    )
    h_c = h_c[0]; r_c = r_c[0]

    # Before
    ax = axes[row, 0]
    ax.plot(t, h, color=HBO_COLOR, linewidth=1.5, label='HbO')
    ax.plot(t, r, color=HBR_COLOR, linewidth=1.5, label='HbR')
    corr_b = np.corrcoef(h, r)[0,1]
    ax.set_title(f'{class_names[label]} — Before (corr={corr_b:+.3f})', fontsize=9)
    if row == 0: ax.legend(fontsize=8)
    ax.set_ylabel(class_names[label]); ax.grid(True)

    # After
    ax = axes[row, 1]
    ax.plot(t, h_c, color=CLEAN_HBO, linewidth=1.5, label='HbO (CBSI)')
    ax.plot(t, r_c, color=CLEAN_HBR, linewidth=1.5, label='HbR (CBSI)')
    corr_a = np.corrcoef(h_c, r_c)[0,1]
    ax.set_title(f'{class_names[label]} — After CBSI (corr={corr_a:+.3f})', fontsize=9)
    if row == 0: ax.legend(fontsize=8)
    ax.grid(True)

for ax in axes[-1]:
    ax.set_xlabel('Time (s)')

plt.tight_layout()
plt.show()

---
## Part 4: Combining ICA + CBSI

The recommended order from Brigadoi et al. (2014):
1. **ICA first** — removes broadband artifacts (motion transients, cardiac spikes)
   that would confuse the CBSI alpha estimation
2. **CBSI second** — removes the remaining systemic hemodynamics using the
   now-cleaner HbO/HbR relationship

This is exactly what a real fNIRS signal processing pipeline does before decoding.


In [ ]:
# ── Run and visualize the full combined pipeline ──

# Use one epoch of each class
results = {}
for label in [0, 1, 2]:
    idx = np.where(proc['y'] == label)[0][0]
    hbo_ep = proc['X_hbo'][idx]
    hbr_ep = proc['X_hbr'][idx]
    hbo_clean, hbr_clean, diag = full_artifact_removal_pipeline(
        hbo_ep, hbr_ep, fs=cfg.fs, return_diagnostics=True
    )
    results[label] = {
        'hbo_raw': hbo_ep, 'hbr_raw': hbr_ep,
        'hbo_clean': hbo_clean, 'hbr_clean': hbr_clean,
        'cbsi_corr_before': diag.get('cbsi', {}).get('mean_corr_before', None) if diag else None,
        'cbsi_corr_after':  diag.get('cbsi', {}).get('mean_corr_after', None) if diag else None,
    }

fig, axes = plt.subplots(3, 3, figsize=(16, 10))
fig.suptitle('Full ICA + CBSI Pipeline — All Three Mental States', fontsize=13)

col_titles = ['Raw HbO signal', 'After ICA + CBSI (HbO)', 'HbO amplitude reduction (systemic noise removed)']
for col, title in enumerate(col_titles):
    axes[0, col].set_title(title, fontsize=9, pad=8)

for row, label in enumerate([0, 1, 2]):
    r = results[label]
    # Average across channels for clarity
    hbo_avg_raw   = r['hbo_raw'].mean(axis=0)
    hbo_avg_clean = r['hbo_clean'].mean(axis=0)
    hbr_avg_raw   = r['hbr_raw'].mean(axis=0)
    hbr_avg_clean = r['hbr_clean'].mean(axis=0)

    # Column 0: raw
    ax = axes[row, 0]
    ax.plot(t, hbo_avg_raw, color=HBO_COLOR, linewidth=1.5, label='HbO raw')
    ax.plot(t, hbr_avg_raw, color=HBR_COLOR, linewidth=1.5, label='HbR raw')
    ax.set_ylabel(class_names[label]); ax.grid(True)
    if row == 0: ax.legend(fontsize=7)

    # Column 1: cleaned
    ax = axes[row, 1]
    ax.plot(t, hbo_avg_clean, color=CLEAN_HBO, linewidth=1.5, label='HbO cleaned')
    ax.plot(t, hbr_avg_clean, color=CLEAN_HBR, linewidth=1.5, label='HbR cleaned')
    ax.grid(True)
    if row == 0: ax.legend(fontsize=7)

    # Column 2: overlay to see what was removed
    ax = axes[row, 2]
    noise_hbo = hbo_avg_raw - hbo_avg_clean
    ax.plot(t, noise_hbo, color='#e74c3c', linewidth=1.2, alpha=0.9, label='Removed noise')
    ax.axhline(0, color='white', linewidth=0.8, linestyle='--')
    ax.grid(True)
    if row == 0: ax.legend(fontsize=7)

for ax in axes[-1]:
    ax.set_xlabel('Time (s)')

plt.tight_layout()
plt.show()

---
## Part 5: Evaluating Cleaning Quality

**The hard problem:** without ground truth (a phantom with known signal + known noise),
how do you know your cleaning actually worked?

Common heuristics used in the fNIRS literature:

| Metric | What it tells you | Expect after cleaning |
|---|---|---|
| HbO-HbR correlation | Systemic noise fraction | More negative |
| Signal variance | How much was removed | Lower |
| Kurtosis | Impulsivity of remaining signal | Lower |
| SNR (vs rest) | Task-related signal preserved | Should be ≥ raw |


In [ ]:
# ── Quantitative evaluation across all epochs ──

metrics = {'raw': [], 'cbsi_only': [], 'ica_cbsi': []}

from utils.artifact_removal import apply_cbsi_dataset

for i in range(min(30, len(proc['y']))):
    hbo_ep = proc['X_hbo'][i]
    hbr_ep = proc['X_hbr'][i]

    # Raw: mean HbO-HbR correlation
    corrs_raw = [np.corrcoef(hbo_ep[c], hbr_ep[c])[0,1] for c in range(hbo_ep.shape[0])]
    metrics['raw'].append(np.mean(corrs_raw))

    # CBSI only
    hbo_c, hbr_c, _ = apply_cbsi(hbo_ep, hbr_ep)
    corrs_cbsi = [np.corrcoef(hbo_c[c], hbr_c[c])[0,1] for c in range(hbo_c.shape[0])]
    metrics['cbsi_only'].append(np.mean(corrs_cbsi))

    # ICA + CBSI
    hbo_f, hbr_f, _ = full_artifact_removal_pipeline(hbo_ep, hbr_ep, fs=cfg.fs)
    corrs_full = [np.corrcoef(hbo_f[c], hbr_f[c])[0,1] for c in range(hbo_f.shape[0])]
    metrics['ica_cbsi'].append(np.mean(corrs_full))

fig, ax = plt.subplots(figsize=(10, 5))
fig.suptitle('HbO-HbR Correlation per Epoch\n(more negative = less systemic noise)', fontsize=12)

x = np.arange(len(metrics['raw']))
ax.plot(x, metrics['raw'],      color=HBO_COLOR,  linewidth=1.5, label='Raw',         marker='o', markersize=3)
ax.plot(x, metrics['cbsi_only'],color='#f39c12',  linewidth=1.5, label='CBSI only',   marker='s', markersize=3)
ax.plot(x, metrics['ica_cbsi'], color=CLEAN_HBR,  linewidth=1.5, label='ICA + CBSI',  marker='^', markersize=3)
ax.axhline(0, color='white', linestyle='--', linewidth=0.8)
ax.axhline(-1, color='#555', linestyle=':', linewidth=0.8)
ax.set_xlabel('Epoch index')
ax.set_ylabel('Mean HbO-HbR correlation')
ax.legend(); ax.grid(True)
ax.set_ylim(-1.1, 1.1)

plt.tight_layout()
plt.show()

for method, vals in metrics.items():
    print(f"{method:12s}: mean corr = {np.mean(vals):+.3f}  (closer to -1.0 = more neural-like)")

---
## Part 6: What to Remember for Temple

### When Sachin asks "how would you handle artifact removal for our single-sensor device?"

**The short answer:**
> "The two main approaches are ICA with wavelet-based virtual channels, and CBSI
> which exploits the anti-correlation between HbO and HbR. In practice you'd use both:
> ICA first to remove broadband motion and cardiac artifacts, then CBSI to remove the
> remaining systemic hemodynamics."

### Key nuances that show depth:

1. **Single-channel ICA is a compromise.** The wavelet decomposition creates mathematically
   independent frequency bands, but they're not truly independent sources. For a real device,
   you'd want even 2–3 sensor pairs to run proper multi-channel ICA.

2. **CBSI forces perfect anti-correlation (corr = −1.0).** This is mathematically guaranteed
   but physically unrealistic. It over-corrects slightly. Hybrid approaches like 
   FICA (fNIRS-ICA) or targeted PCA can preserve more signal variance.

3. **Kurtosis-based artifact identification is simple but works.** More sophisticated:
   train a small classifier on component features (kurtosis, skewness, spectral slope,
   frequency of peak power). Real fNIRS pipelines like MNE-NIRS and Homer3 do this.

4. **The evaluation problem is hard.** Without a phantom (a physical object with known
   optical properties and known "ground truth" neural signal), you can't measure
   cleaning accuracy directly. You evaluate proxies: HbO/HbR correlation, SNR vs rest,
   classification accuracy downstream.

5. **Temple's constraint is real.** No short-separation channel = software-only cleaning.
   This is an open research problem — any approach you propose for improving it would
   genuinely matter to the team.

### References
- Brigadoi et al. (2014): "Motion artifacts in fNIRS: a comparison of motion correction techniques"
- Cui et al. (2010): "CBSI — functional NIRS signal improvement based on negative correlation"
- Hyvarinen & Oja (2000): "ICA: algorithms and applications" (the FastICA paper)
- Scholkmann et al. (2014): "A review on continuous wave fNIRS methodology"
